In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

pd.set_option('display.max_columns', None)
# pd.reset_option('display.max_columns')

sns.set_theme(style='darkgrid')
# plt.rcdefaults() 

loan_data = pd.read_csv('loan_data_2007_2014.csv', index_col=0, low_memory=False)

In [2]:
# Preprocessing raw data

loan_data['emp_length_int'] = loan_data['emp_length'].str.replace('+ years', '', regex=False)
loan_data['emp_length_int'] = loan_data['emp_length_int'].str.replace('< 1 year', '0')
loan_data['emp_length_int'] = loan_data['emp_length_int'].str.replace(' years', '')
loan_data['emp_length_int'] = loan_data['emp_length_int'].str.replace(' year', '')
loan_data['emp_length_int'] = loan_data['emp_length_int'].fillna('0')
loan_data['emp_length_int'] = loan_data['emp_length_int'].astype('uint8')

loan_data['term_int'] = loan_data['term'].str.replace(' months','')
loan_data['term_int'] = loan_data['term_int'].str.replace(' ','').astype('uint8')

loan_data['earliest_cr_line_date'] = pd.to_datetime(loan_data['earliest_cr_line'], format='%b-%y')
loan_data['mths_since_earliest_cr_line'] = np.floor((pd.to_datetime('2026-01-01') - loan_data['earliest_cr_line_date']).dt.days/30.44)
# loan_data['mths_since_earliest_cr_line'].describe()
loan_data['earliest_cr_line_date'] = np.where(loan_data['earliest_cr_line_date'] > pd.to_datetime('2026-01-01'), 
                                              loan_data['earliest_cr_line_date'] - pd.DateOffset(years=100), loan_data['earliest_cr_line_date'])
loan_data['mths_since_earliest_cr_line'] = np.floor((pd.to_datetime('2026-01-01') - loan_data['earliest_cr_line_date']).dt.days/30.44)
# loan_data['mths_since_earliest_cr_line'].describe()

loan_data['issue_d'] = pd.to_datetime(loan_data['issue_d'], format='%b-%y')
loan_data['mths_since_issue_d'] = np.floor((pd.to_datetime('2026-01-01') - loan_data['issue_d']).dt.days/30.44)
# loan_data['mths_since_issue_d'].describe()

# splitting variables into dummy variables, amending types to int8 instead of booleans
col_names = ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'loan_status', 'purpose', 'addr_state', 'initial_list_status']
loan_data_dummies = []
for name in col_names:
    dummy = pd.get_dummies(loan_data[name], dtype='int8', prefix=name, prefix_sep=':')
    loan_data_dummies.append(dummy)

loan_data_dummies = pd.concat(loan_data_dummies, axis=1)
loan_data = pd.concat([loan_data, loan_data_dummies], axis=1)

# Checking null values
# null = loan_data.isnull().sum()
# null[null > 0]
loan_data['total_rev_hi_lim'] = loan_data['total_rev_hi_lim'].fillna(loan_data['funded_amnt'])

loan_data['annual_inc'] = loan_data['annual_inc'].fillna(loan_data['annual_inc'].mean())

cols = ['delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'total_acc', 'acc_now_delinq', 'emp_length_int', 'mths_since_earliest_cr_line']
loan_data[cols] = loan_data[cols].fillna(0)

# Dependent variable preparation
# loan_data['loan_status'].value_counts()
loan_data['good_bad'] = np.where(loan_data['loan_status'].isin(['Charged Off', 'Late (31-120 days)', 'Default', 'Does not meet the credit policy. Status:Charged Off']), 0, 1)

In [3]:
# Preparing variables before calculating LGD & EAD

loan_data_defaults = loan_data[loan_data['loan_status'].isin(['Charged Off', 'Does not meet the credit policy. Status:Charged Off'])].copy()
loan_data_defaults.isnull().sum().to_frame().T
# Only going to use these two columns
loan_data_defaults['mths_since_last_delinq'] = loan_data_defaults['mths_since_last_delinq'].fillna(0)
loan_data_defaults['mths_since_last_record'] = loan_data_defaults['mths_since_last_record'].fillna(0)
loan_data_defaults.sort_index(axis=1, inplace=True)

# LGD = 1 - recovery_rate
loan_data_defaults['recovery_rate'] = loan_data_defaults['recoveries'] / loan_data_defaults['funded_amnt']
loan_data_defaults['recovery_rate'] = np.where(loan_data_defaults['recovery_rate'] > 1, 1, loan_data_defaults['recovery_rate'])
loan_data_defaults['recovery_rate'] = np.where(loan_data_defaults['recovery_rate'] < 0, 0, loan_data_defaults['recovery_rate'])

# EAD <- CCF - credit conversion factor
loan_data_defaults['CCF'] = (loan_data_defaults['funded_amnt'] - loan_data_defaults['total_rec_prncp']) / loan_data_defaults['funded_amnt']

# loan_data_defaults.to_csv('loan_data_defaults.csv')

In [4]:
# LGD Stage 1 - is the recovery rate equal to or greater than zero?

# Beta regression is used to model results that are strictly greater than zero and strictly less than one
# However, this function is not available in Python. You can import the R package, but it requires a lot of memory and may not be able to handle it on private computers

# plt.hist(loan_data_defaults['recovery_rate'], bins=50)
# plt.hist(loan_data_defaults['CCF'], bins=50)

# LGD: To estimate if recovery_rate is greater than 0 we use logistic regression -> if it is then we estimate recovery_rate by linear regression, if it is not greater then recovery rate = 0
loan_data_defaults['recovery_rate_0_1'] = np.where(loan_data_defaults['recovery_rate'] > 0, 1, 0)

#EAD: To calculate EAD via CCF, linear regression can be used directly, due to better distribution of variables (it is not 0/1)

# we remove the variables that we know for sure are dependent variables of one of the models, recovery_0_1 as the target because I check if it will be 0 or 1
lgd_inputs_train_1, lgd_inputs_test_1, lgd_targets_train_1, lgd_targets_test_1 = train_test_split(
    loan_data_defaults.drop(['good_bad', 'recovery_rate', 'recovery_rate_0_1', 'CCF'], axis=1), loan_data_defaults['recovery_rate_0_1'],
    test_size=0.2, random_state=42, stratify=loan_data_defaults['recovery_rate_0_1'])

In [5]:
# FEATURES

features_all = [
'grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'grade:F',
'grade:G',
'home_ownership:MORTGAGE',
'home_ownership:NONE',
'home_ownership:OTHER',
'home_ownership:OWN',
'home_ownership:RENT',
'verification_status:Not Verified',
'verification_status:Source Verified',
'verification_status:Verified',
'purpose:car',
'purpose:credit_card',
'purpose:debt_consolidation',
'purpose:educational',
'purpose:home_improvement',
'purpose:house',
'purpose:major_purchase',
'purpose:medical',
'purpose:moving',
'purpose:other',
'purpose:renewable_energy',
'purpose:small_business',
'purpose:vacation',
'purpose:wedding',
'initial_list_status:f',
'initial_list_status:w',
'term_int',
'emp_length_int',
'mths_since_issue_d',
'mths_since_earliest_cr_line',
'funded_amnt',
'int_rate',
'installment',
'annual_inc',
'dti',
'delinq_2yrs',
'inq_last_6mths',
'mths_since_last_delinq',
'mths_since_last_record',
'open_acc',
'pub_rec',
'total_acc',
'acc_now_delinq',
'total_rev_hi_lim'
]

features_ref_categories = [
'grade:G',
'home_ownership:RENT',
'verification_status:Verified',
'purpose:credit_card',
'initial_list_status:f'
]

lgd_inputs_train_1 = lgd_inputs_train_1[features_all].drop(features_ref_categories, axis=1)

In [6]:
# Create a new fit function to calculate p-values ​​for LogisticRegression

from sklearn import linear_model
import scipy.stats as stat

class LogisticRegression_with_p_values:
# this part copies what the LogRef function has
    def __init__(self,*args, **kwargs):
        self.model = linear_model.LogisticRegression(*args, **kwargs)
    
# override original fit function
    def fit(self, X, y):
        self.model.fit(X,y)
        denom = (2.0 * (1.0 + np.cosh(self.model.decision_function(X))))
        denom = np.tile(denom, (X.shape[1],1)).T
        # adding Ridge Penalty to prevent Singular Matrix error
        F_ij = np.dot((X/denom).T,X) + 1e-6
        Cramer_Rao = np.linalg.inv(F_ij)
        sigma_estimates = np.sqrt(np.diagonal(Cramer_Rao))
        z_scores = self.model.coef_[0] / sigma_estimates
        p_values = [stat.norm.sf(abs(x)) * 2 for x in z_scores]
        self.coef_ = self.model.coef_
        self.intercept_ = self.model.intercept_
        self.p_values = p_values
        return

In [7]:
# Building new table with coefficients and intercept for regression

reg_lgd_1 = LogisticRegression_with_p_values(random_state=42, max_iter=5000)
reg_lgd_1.fit(lgd_inputs_train_1, lgd_targets_train_1.values.ravel())

feature_name = lgd_inputs_train_1.columns.values

summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
summary_table['Coefficients'] = np.transpose(reg_lgd_1.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg_lgd_1.intercept_[0]]
summary_table = summary_table.sort_index()
summary_table['p_values'] = np.append(np.nan, np.array(reg_lgd_1.p_values))

c:\Users\Mateusz\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
#Checking accuracy of the model

lgd_inputs_test_1 = lgd_inputs_test_1[features_all].drop(features_ref_categories, axis=1)

y_hat_test_lgd_1 = reg_lgd_1.model.predict(lgd_inputs_test_1)
y_hat_test_proba_lgd_1 = reg_lgd_1.model.predict_proba(lgd_inputs_test_1)
y_hat_test_proba_lgd_1 = y_hat_test_proba_lgd_1[:, 1]

df_actual_predicted_probs = pd.DataFrame({
    'lgd_targets_test_1': lgd_targets_test_1.reset_index(drop=True),
    'y_hat_test_proba': y_hat_test_proba_lgd_1,
    'y_hat_test': y_hat_test_lgd_1
    })

# Creating column indicating if recovery is greater than 0
df_actual_predicted_probs['y_hat_test'] = np.where(df_actual_predicted_probs['y_hat_test_proba'] > 0.5, 1, 0)
# Creating confusion matrix to check the accuracy
pd.crosstab(df_actual_predicted_probs['lgd_targets_test_1'], df_actual_predicted_probs['y_hat_test'], rownames=['Actual'], colnames=['Predicted'])
print(f'Accuracy:',(pd.crosstab(df_actual_predicted_probs['lgd_targets_test_1'], df_actual_predicted_probs['y_hat_test']).iloc[0,0] +\
                    pd.crosstab(df_actual_predicted_probs['lgd_targets_test_1'], df_actual_predicted_probs['y_hat_test']).iloc[1,1]) / df_actual_predicted_probs.shape[0])

Accuracy: 0.6300878815911193


In [9]:
# AUROC

from sklearn.metrics import roc_curve, roc_auc_score

# fpr, tpr, tresholds = roc_curve(df_actual_predicted_probs['lgd_targets_test_1'], df_actual_predicted_probs['y_hat_test_proba'])
# plt.plot(fpr, tpr)
# plt.plot(fpr, fpr, linestyle='--', color='k')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve')
# plt.show()

AUROC = roc_auc_score(df_actual_predicted_probs['lgd_targets_test_1'], df_actual_predicted_probs['y_hat_test_proba'])
print(f'AUROC:',AUROC)

AUROC: 0.656749508994407


In [ ]:
# Saving lgd_model_1

import pickle

# wb to write bites, przy wczytywaniu daje sie rb, 'dump' = save, 
pickle.dump(reg_lgd_1, open('lgd_model_1.sav', 'wb'))

In [11]:
# LGD Stage 2 - estimating recovery rate for values greater than 0

lgd_data_2 = loan_data_defaults[loan_data_defaults['recovery_rate_0_1'] == 1]

# recovery rate because now I check recovery rate > 0, stratify works only on discrete data, not continoues
lgd_inputs_train_2, lgd_inputs_test_2, lgd_targets_train_2, lgd_targets_test_2 = train_test_split(
    lgd_data_2.drop(['good_bad', 'recovery_rate', 'recovery_rate_0_1', 'CCF'], axis=1), lgd_data_2['recovery_rate'],
    test_size=0.2, random_state=42)

In [12]:
# Override LinearRegression function

import scipy.stats as stat

class LinearRegression(linear_model.LinearRegression):
    def fit(self, X, y):
        super().fit(X, y)
        sse = np.sum((self.predict(X) - y) ** 2, axis=0) / float(X.shape[0] - X.shape[1])
        se = np.array([np.sqrt(np.diagonal(sse * np.linalg.inv(np.dot(X.T, X))))])
        self.t_values = self.coef_ / se               
        self.p_values = np.squeeze(2 * (1 - stat.t.cdf(np.abs(self.t_values), y.shape[0] - X.shape[1]))) 
        return self

In [13]:
# Building a NEW table with coefficients and intercept for regression based on filtered data

lgd_inputs_train_2 = lgd_inputs_train_2[features_all].drop(features_ref_categories, axis=1)

reg_lgd_2 = LinearRegression()
reg_lgd_2.fit(lgd_inputs_train_2, lgd_targets_train_2)

feature_name = lgd_inputs_train_2.columns.values

summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
summary_table['Coefficients'] = np.transpose(reg_lgd_2.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg_lgd_2.intercept_]
summary_table = summary_table.sort_index()
summary_table['p_values'] = np.append(np.nan, np.array(reg_lgd_2.p_values).round(3))

In [15]:
# Checking the quality of the model by correlations and saving the model

lgd_inputs_test_2 = lgd_inputs_test_2[features_all].drop(features_ref_categories, axis=1)
y_hat_test_lgd_2 = reg_lgd_2.predict(lgd_inputs_test_2)

pd.DataFrame(
    {'lgd_targets_test_2': lgd_targets_test_2.reset_index(drop=True), 
     'y_hat_test': y_hat_test_lgd_2}).corr()

# when the mean is close to zero and most of the data is around this, it is a good model
# sns.displot(lgd_targets_test_2 - y_hat_test_lgd_2)

,lgd_targets_test_2,y_hat_test
lgd_targets_test_2,1.000000,0.307852
y_hat_test,0.307852,1.000000


In [ ]:
# Saving lgd_model2

# wb to write bites, when loading give rb, 'dump' = save
pickle.dump(reg_lgd_2, open('lgd_model2.sav', 'wb'))

In [17]:
# Combining Stage 1 and Stage 2 - predicting recovery rate

y_hat_test_lgd_2 = reg_lgd_2.predict(lgd_inputs_test_1)
y_hat_test_lgd = y_hat_test_lgd_1 * y_hat_test_lgd_2
pd.DataFrame(y_hat_test_lgd).describe()

y_hat_test_lgd = np.where(y_hat_test_lgd < 0, 0, y_hat_test_lgd)
y_hat_test_lgd = np.where(y_hat_test_lgd > 1, 1, y_hat_test_lgd)
pd.DataFrame(y_hat_test_lgd).describe()

,0
count,8648.000000
mean,0.076627
std,0.054342
min,0.000000
25%,0.000000
50%,0.094148
75%,0.119116
max,0.266677


In [18]:
# EAD Model - Estimation and Interpretation

ead_inputs_train, ead_inputs_test, ead_targets_train, ead_targets_test = train_test_split(loan_data_defaults.drop(['good_bad', 'recovery_rate', 'recovery_rate_0_1', 'CCF'], axis=1), 
                                                                                          loan_data_defaults['CCF'], test_size=0.2, random_state=42)

ead_inputs_train = ead_inputs_train[features_all].drop(features_ref_categories, axis=1)

reg_ead = LinearRegression()
reg_ead.fit(ead_inputs_train, ead_targets_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [ ]:
# Building a NEW table with coefficients and intercept for regression based on filtered data

feature_name = ead_inputs_train.columns.values

summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
summary_table['Coefficients'] = np.transpose(reg_ead.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg_ead.intercept_]
summary_table = summary_table.sort_index()
summary_table['p_values'] = np.append(np.nan, np.array(reg_ead.p_values).round(3))

ead_inputs_test = ead_inputs_test[features_all].drop(features_ref_categories, axis=1)
y_hat_test_ead = reg_ead.predict(ead_inputs_test)

In [20]:
# Checking the quality of the model by correlations

pd.DataFrame(
    {'ead_targets_test': ead_targets_test.reset_index(drop=True), 
     'y_hat_test': y_hat_test_ead}
     ).corr()

# when the mean is close to zero and most of the data is around this, it is a good model
# sns.displot(ead_targets_test - y_hat_test_ead)

# maks i min from 0 to 1
y_hat_test_ead = np.where(y_hat_test_ead < 0, 0, y_hat_test_ead)
y_hat_test_ead = np.where(y_hat_test_ead > 1, 1, y_hat_test_ead)
pd.DataFrame(y_hat_test_ead).describe()

,0
count,8648.000000
mean,0.735971
std,0.105056
min,0.383687
25%,0.661278
50%,0.731624
75%,0.811175
max,1.000000


In [21]:
# Expected loss; LGD & EAD

loan_data['mths_since_last_delinq'] = loan_data['mths_since_last_delinq'].fillna(0)
loan_data['mths_since_last_record'] = loan_data['mths_since_last_record'].fillna(0)

loan_data_lgd_ead = loan_data[features_all].drop(features_ref_categories, axis=1)

# LGD
loan_data['recovery_rate_1'] = reg_lgd_1.model.predict(loan_data_lgd_ead)
loan_data['recovery_rate_2'] = reg_lgd_2.predict(loan_data_lgd_ead)
loan_data['recovery_rate'] = loan_data['recovery_rate_1'] * loan_data['recovery_rate_2']
loan_data['recovery_rate'] = np.where(loan_data['recovery_rate'] > 1, 1, loan_data['recovery_rate'])
loan_data['recovery_rate'] = np.where(loan_data['recovery_rate'] < 0, 0, loan_data['recovery_rate'])
loan_data['LGD'] = 1 - loan_data['recovery_rate']

# EAD
loan_data['CCF'] = reg_ead.predict(loan_data_lgd_ead)
loan_data['CCF'] = np.where(loan_data['CCF'] < 0, 0, loan_data['CCF'])
loan_data['CCF'] = np.where(loan_data['CCF'] > 1, 1, loan_data['CCF'])

# added funded_amnt to features_with_ref_cat
loan_data['EAD'] = loan_data['CCF'] * loan_data_lgd_ead['funded_amnt']

In [22]:
# Preparing the data for Expected Loss

loan_data_inputs_train = pd.read_csv('loan_data_inputs_train.csv')
loan_data_inputs_test = pd.read_csv('loan_data_inputs_test.csv')

loan_data_inputs_pd = pd.concat([loan_data_inputs_train, loan_data_inputs_test], axis=0).set_index('Unnamed: 0')

In [23]:
# Features for PD

features_all_pd = [
    'grade:A',
    'grade:B',
    'grade:C',
    'grade:D',
    'grade:E',
    'grade:F',
    'grade:G',
    'home_ownership:RENT_OTHER_NONE_ANY',
    'home_ownership:OWN',
    'home_ownership:MORTGAGE',
    'addr_state:NE_IA_NV_FL_HI_AL',
    'addr_state:NM_VA',
    'addr_state:NY',
    'addr_state:OK_TN_MO_LA_MD_NC',
    'addr_state:CA',
    'addr_state:UT_KY_AZ_NJ',
    'addr_state:AR_MI_PA_OH_MN',
    'addr_state:RI_MA_DE_SD_IN',
    'addr_state:GA_WA_OR',
    'addr_state:WI_MT',
    'addr_state:TX',
    'addr_state:IL_CT',
    'addr_state:KS_SC_CO_VT_AK_MS',
    'addr_state:WV_NH_WY_DC_ME_ID',
    'verification_status:Not Verified',
    'verification_status:Source Verified',
    'verification_status:Verified',
    'purpose:educ__sm_b__wedd__ren_en__mov__house',
    'purpose:credit_card',
    'purpose:debt_consolidation',
    'purpose:oth__med__vacation',
    'purpose:major_purch__car__home_impr',
    'initial_list_status:f',
    'initial_list_status:w',
    'term_int:36',
    'term_int:60',
    'emp_length_int:0',
    'emp_length_int:1',
    'emp_length_int:2-4',
    'emp_length_int:5-6',
    'emp_length_int:7-9',
    'emp_length_int:10',
    'mths_since_issue_d:<135',
    'mths_since_issue_d:135-136',
    'mths_since_issue_d:137-138',
    'mths_since_issue_d:139-145',
    'mths_since_issue_d:146-149',
    'mths_since_issue_d:150-161',
    'mths_since_issue_d:162-181',
    'mths_since_issue_d:>181',
    'int_rate:<9.548',
    'int_rate:9.548-12.025',
    'int_rate:12.025-15.74',
    'int_rate:15.74-20.281',
    'int_rate:>20.281',
    'mths_since_earliest_cr_line:<237',
    'mths_since_earliest_cr_line:238-261',
    'mths_since_earliest_cr_line:262-344',
    'mths_since_earliest_cr_line:345-367',
    'mths_since_earliest_cr_line:368-449',
    'mths_since_earliest_cr_line:>449',
    # 'delinq_2yrs:0',
    # 'delinq_2yrs:1-3',
    # 'delinq_2yrs:>3',
    'inq_last_6mths:0',
    'inq_last_6mths:1-2',
    'inq_last_6mths:3-6',
    'inq_last_6mths:>6',
    # 'open_acc:0',
    # 'open_acc:1-3',
    # 'open_acc:4-12',
    # 'open_acc:13-17',
    # 'open_acc:18-22',
    # 'open_acc:23-25',
    # 'open_acc:26-30',
    # 'open_acc:>30',
    # 'pub_rec:0-2',
    # 'pub_rec:3-4',
    # 'pub_rec:>4',
    # 'total_acc:<=27',
    # 'total_acc:28-51',
    # 'total_acc:>51',
    'acc_now_delinq:0',
    'acc_now_delinq:>0',
    # 'total_rev_hi_lim:<=5K',
    # 'total_rev_hi_lim:5K-10K',
    # 'total_rev_hi_lim:10K-20K',
    # 'total_rev_hi_lim:20K-30K',
    # 'total_rev_hi_lim:30K-40K',
    # 'total_rev_hi_lim:40K-55K',
    # 'total_rev_hi_lim:55K-95K',
    # 'total_rev_hi_lim:>95K',
    'annual_inc:<20K',
    'annual_inc:20K-30K',
    'annual_inc:30K-40K',
    'annual_inc:40K-50K',
    'annual_inc:50K-60K',
    'annual_inc:60K-70K',
    'annual_inc:70K-80K',
    'annual_inc:80K-90K',
    'annual_inc:90K-100K',
    'annual_inc:100K-120K',
    'annual_inc:120K-140K',
    'annual_inc:>140K',
    'dti:<=1.4',
    'dti:1.4-3.5',
    'dti:3.5-7.7',
    'dti:7.7-10.5',
    'dti:10.5-16.1',
    'dti:16.1-20.3',
    'dti:20.3-21.7',
    'dti:21.7-22.4',
    'dti:22.4-35',
    'dti:>35',
    'mths_since_last_delinq:N/A',
    'mths_since_last_delinq:0-3',
    'mths_since_last_delinq:4-30',
    'mths_since_last_delinq:31-56',
    'mths_since_last_delinq:>56',
    'mths_since_last_record:N/A',
    'mths_since_last_record:0-2',
    'mths_since_last_record:3-20',
    'mths_since_last_record:21-31',
    'mths_since_last_record:32-80',
    'mths_since_last_record:81-86',
    'mths_since_last_record:>86',
    ]

ref_categories_pd = [
    'grade:G',
    'home_ownership:RENT_OTHER_NONE_ANY',
    'addr_state:NE_IA_NV_FL_HI_AL',
    'verification_status:Verified',
    'purpose:educ__sm_b__wedd__ren_en__mov__house',
    'initial_list_status:f',
    'term_int:60',
    'emp_length_int:0',
    'mths_since_issue_d:>181',
    'int_rate:>20.281',
    'mths_since_earliest_cr_line:<237',
    #'delinq_2yrs:>3',
    'inq_last_6mths:>6',
    # 'open_acc:0',
    # 'pub_rec:0-2',
    # 'total_acc:<=27',
    'acc_now_delinq:0',
    # 'total_rev_hi_lim:<=5K',
    'annual_inc:<20K',
    'dti:>35',
    'mths_since_last_delinq:0-3',
    'mths_since_last_record:0-2'
]

loan_data_inputs_pd = loan_data_inputs_pd[features_all_pd].drop(ref_categories_pd, axis=1)

In [ ]:
# Applying PD model

import pickle
reg_pd = pickle.load(open('pd_model.sav', 'rb'))

loan_data_inputs_pd['PD'] = pd.DataFrame(reg_pd.model.predict_proba(loan_data_inputs_pd)).iloc[:,0]

In [56]:
# Calculating Expected Loss

loan_data_preprocessed = pd.concat([loan_data, loan_data_inputs_pd], axis=1)

loan_data_preprocessed['EL'] = loan_data_preprocessed['PD'] * loan_data_preprocessed['LGD'] * loan_data_preprocessed['EAD']
loan_data_preprocessed[['funded_amnt', 'PD', 'LGD', 'EAD', 'EL']].head()

C:\Users\Mateusz\AppData\Local\Temp\ipykernel_12540\1537665930.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  loan_data_preprocessed['EL'] = loan_data_preprocessed['PD'] * loan_data_preprocessed['LGD'] * loan_data_preprocessed['EAD']


,funded_amnt,PD,LGD,EAD,EL
0,5000,0.024530,0.914481,2943.645681,66.031728
1,2500,0.201715,0.915937,1941.597608,358.726548
2,2400,0.177048,0.920014,1576.360227,256.768267
3,10000,0.038025,0.905243,6599.043968,227.151316
4,3000,0.099924,1.000000,2121.583276,211.997109


In [57]:
print(f'Expected Loss on the entire portfolio:',round(loan_data_preprocessed['EL'].sum() / loan_data_preprocessed['funded_amnt'].sum() * 100,2),'%')

Expected Loss on the entire portfolio: 7.68 %
